### RAG Pipeline DATA INGESTION to Vector DB 

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
### Read all the pdf inside the directory
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents=[]
    pdf_dirs=Path(pdf_directory)

    #find allpdf files recursively
    pdf_files=list(pdf_dirs.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} pdf files to be processed")

    for pdf_file in pdf_files:
        print(f"\n Processing:{pdf_file.name}")

        try:
            loader=PyMuPDFLoader(str(pdf_file))
            documents=loader.load()
            #Add source info

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata["file_type"]='pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)}pages")
        except Exception as e:
            print(f" Error:{e}")
    print(f"\n total documents loaded:{len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data")



found 1 pdf files to be processed

 Processing:AJP U3 Notes.pdf
 Loaded 52pages

 total documents loaded:52


In [4]:
### Split the documents into smaller chunks
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def split_documents(documents,chunk_size=500,chunk_overlap=200):
    """split documents into smaller chunks"""

    for doc in documents:
        doc.page_content = clean_text(doc.page_content)

    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,
       length_function=len,
       separators=["\n\n", "\n", " ", "",". "] 
       )
    split_docs=text_splitter.split_documents(documents)
    print(f"split{len(documents)} document into {len(split_docs)} chunks")

    if split_docs:
        print(f"example chunk:\n {split_docs[0].page_content[:200]}...")
        print(f"metadata:{split_docs[0].metadata}")
    return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)

split52 document into 111 chunks
example chunk:
 Introduction to JSP JSP stands for Java Server Pages. It is a server-side technology which is used for creating web applications. It is used to create dynamic web content. JSP consists of both HTML ta...
metadata:{'producer': 'Skia/PDF m125 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\text_files\\AJP U3 Notes.pdf', 'file_path': '..\\data\\text_files\\AJP U3 Notes.pdf', 'total_pages': 52, 'format': 'PDF 1.4', 'title': 'AJP U3 Notes', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'AJP U3 Notes.pdf', 'file_type': 'pdf'}


### embedding and vector store db

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        self.model = SentenceTransformer(self.model_name)
        print(f"embedding dimension:{self.model.get_embedding_dimension()}")

    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """Generate embeddings for a list of texts"""
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings for {len(texts)} texts")
        print(f"embedding shape:{embeddings.shape}")
        return embeddings

        ##initialise embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3111.50it/s]


embedding dimension:384


In [8]:
### vector store using chromadb
class VectorStore:
    def __init__(self,collection_name:str="pdf_embeddings",persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            
            self.collection=self.client.get_or_create_collection(name=self.collection_name,metadata={
                "hnsw:space": "cosine",
                "description":"Collection of PDF document embeddings"}
                                                                
            )
            print(f"Initialized vector store with collection name: {self.collection_name}")
            print(f"existing documents in store:{self.collection.count()}")
           
        except Exception as e:

            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """Add documents and their embeddings to the vector store"""
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding ) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(ids=ids,embeddings=embeddings_list,metadatas=metadatas,documents=documents_text)
            print(f"Added {len(documents)} documents to vector store")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise 
vector_store=VectorStore()
vector_store

Initialized vector store with collection name: pdf_embeddings
existing documents in store:111


In [9]:
### convert text to embeddings
texts=[doc.page_content for doc in chunks]

## generate embeddings

embeddings=embedding_manager.generate_embeddings(texts)

### store in vector database

vector_store.add_documents(chunks,embeddings)

Batches: 100%|██████████| 4/4 [00:03<00:00,  1.13it/s]

Generated embeddings for 111 texts
embedding shape:(111, 384)
Added 111 documents to vector store


### RAG Retrieval pipeline

In [10]:
class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
      

    def retrieve(self,
                 query:str,
                 top_k:int=5,
                 score_threshold:float=0.0)->List[Dict[str,Any]]:
        """Retrieve relevant documents for a given query"""
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]
        try:
            results=self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()],n_results=top_k)

            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                for i, (doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):

                    similarity_score=1-distance

                    if similarity_score>=score_threshold:
                     retrieved_docs.append({
                        "id":doc_id,
                        "content":document,
                        "metadata":metadata,
                        "similarity_score":similarity_score,
                        "distance":distance,
                        "rank":i+1
                    })

                print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise

rag_retriever=RAGRetriever(vector_store,embedding_manager)

In [10]:
rag_retriever.retrieve("Introduction to JSP")

Batches: 100%|██████████| 1/1 [00:00<00:00, 49.51it/s]

Generated embeddings for 1 texts
embedding shape:(1, 384)
Retrieved 5 documents for query: 'Introduction to JSP'


[{'id': 'doc_9d6692e5_24',
  'content': 'a Web-based technology that helps us to create dynamic and platform-independent web pages. In this, Java code can be inserted in HTML/ XML pages or both. JSP is first converted into a servlet by JSP container before processing the client’s request. JSP Processing is illustrated and discussed in sequential steps prior to which a pictorial media is provided as a handful pick to understand the JSP processing better which is as follows: Step 1: The client navigates to a file ending with the .jsp',
  'metadata': {'modDate': '',
   'author': '',
   'file_type': 'pdf',
   'keywords': '',
   'creator': '',
   'doc_index': 24,
   'source_file': 'AJP U3 Notes.pdf',
   'content_length': 496,
   'trapped': '',
   'format': 'PDF 1.4',
   'total_pages': 52,
   'file_path': '..\\data\\text_files\\AJP U3 Notes.pdf',
   'title': 'AJP U3 Notes',
   'producer': 'Skia/PDF m125 Google Docs Renderer',
   'source': '..\\data\\text_files\\AJP U3 Notes.pdf',
   'moddate

### Integration VectorDB COntext pipeline with LLM output


In [16]:
### simple RAG pipeline with groq LLM
 
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv  
load_dotenv()  

### initialize groq llm

groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## Simple RAG function (retieve context + generate response)
def rag_simple(query,retriever,llm,top_k=3):

    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        print("No relevant documents found. Generating response without context.")
        
    ## generayte response using the llm

    prompt=f"""Use the following context to answer the question concisely.
        Context:\n{context}
        Question:{query}
        Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [ ]:
answer=rag_simple("What is taglib in JSP?",rag_retriever,llm)
print("Answer:",answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 51.58it/s]


Generated embeddings for 1 texts
embedding shape:(1, 384)
Retrieved 3 documents for query: 'What is taglib in JSP?'
Answer: In JSP, a taglib (tag library) is a collection of custom tags that can be used to extend the functionality of JSP pages. It is defined in a .tld (Tag Library Descriptor) file and is used to map custom tags to their corresponding Java classes. The taglib directive is used to import a tag library into a JSP page, allowing the use of its custom tags.


### Enhanced RAG pipeline

In [18]:
def rag_advanced(query,retriever,llm,top_k=3,min_score=0.2,return_context=False):

    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k,score_threshold=min_score)
    if not results:
        return{'answer': "No relevant documents found above the score threshold. Generating response without context.",'sources':[],'confidence':0.0,'context':''}
     

    
    
    context="\n\n".join([doc['content'] for doc in results]) 
    sources=[{
        'sources':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:200] + "..."
    } for doc in results]
    confidence=max(doc['similarity_score'] for doc in results)
        
    ## generayte response using the llm

    prompt=f"""Use the following context to answer the question concisely.
        Context:\n{context}
        Question:{query}
        Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])

    output={
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context']=context
    return output

In [19]:
result=rag_advanced("What is taglib in JSP?",rag_retriever,llm,top_k=3,min_score=0.2,return_context=True)
print("Answer:",result['answer'])
print("\nSources:",result['sources'])
print("\nConfidence:",result['confidence'])
print("\nContext:",result['context'][:300])

Batches: 100%|██████████| 1/1 [00:00<00:00, 22.46it/s]


Generated embeddings for 1 texts
embedding shape:(1, 384)
Retrieved 3 documents for query: 'What is taglib in JSP?'
Answer: In JSP, a taglib (tag library) is a collection of custom tags that can be used to extend the functionality of JSP pages. It is defined in a .tld (Tag Library Descriptor) file and is used to map custom tags to their corresponding Java classes. The taglib directive is used to import a tag library into a JSP page, allowing the use of its custom tags.

Sources: [{'sources': 'AJP U3 Notes.pdf', 'page': 50, 'score': 0.7070754766464233, 'preview': '7. 8. <jsp-config> 9. <taglib> 10.<taglib-uri>mytags</taglib-uri> 11.<taglib-location>/WEB-INF/mytags.tld</taglib-location> 12.</taglib> 13.</jsp-config> 14. 15.</web-app> mytags.tld 1. <?xml version=...'}, {'sources': 'AJP U3 Notes.pdf', 'page': 50, 'score': 0.7070754766464233, 'preview': '7. 8. <jsp-config> 9. <taglib> 10.<taglib-uri>mytags</taglib-uri> 11.<taglib-location>/WEB-INF/mytags.tld</taglib-location> 12.</taglib> 1

In [25]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is directives in jsp?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Batches: 100%|██████████| 1/1 [00:00<00:00, 51.29it/s]

Generated embeddings for 1 texts
embedding shape:(1, 384)
Retrieved 3 documents for query: 'what is directives in jsp?'
Streaming answer:
Use the following context to answer the question concisely.
Context:
Syntax of JSP Directive 1. <%@ directive attribute="value" %> JSP page directive

Syntax of JSP Directive 1. <%@ directive attribute="value" %> JSP page directive

%> 2. <html> 3. <body> 4. 5. Sorry following exception occured:<%= exception %> 6. 7. <

/body> 8. </html> JSP directives The jsp directives are messages that tells the web container how to translate a JSP page into the corresponding servlet. There are three types of directives: ○ page directive ○ include directive ○ taglib directive

Question: what is directives in jsp?

Answer:

Final Answer: JSP directives are messages that tell the web container how to translate a JSP page into the corresponding servlet. There are three types: page, include, and taglib directives.

Citations:
[1] AJP U3 Notes.pdf (page 26)
[2] AJP U3 Notes.pdf (page 26)
[3] AJP U3 Notes.pdf (page 25)
Summary: JSP directives are instructions that inform the web container on how to translate a JSP page into a servlet. There are three main types of JSP directives: page, include, and taglib directives, which serve distinct purposes in the translation process.
History: {'question': 'what is directives in jsp?', 'answer': 'JSP directives are messages that tell the web container how to translate a JSP page in